In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'cluster':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from data_provider.data_processing import (
    clean_dataframe_columns, get_common_sites,
    align_two_site_series, reconstruction_bounds, split_sites_fivefold,
)

SEQ_LEN = 32
QUERY_RATIO = 0.34
SEED = 2030


In [ ]:
def cluster_sites(x_path, y_path, time_step=1):
    x = clean_dataframe_columns(pd.read_csv(PROJECT_ROOT / x_path)).sort_values('year')
    y = clean_dataframe_columns(pd.read_csv(PROJECT_ROOT / y_path)).sort_values('year')
    sites = get_common_sites(x, y)
    histories = []
    for site in sorted(sites):
        years, values, _ = align_two_site_series(x, y, site)
        if not np.all(np.diff(years) == time_step):
            raise ValueError(f'{site}: aligned record contains time gaps')
        _, query_start = reconstruction_bounds(len(values), SEQ_LEN, QUERY_RATIO)
        support = values[:query_start].reshape(-1)
        histories.append((support - support.mean()) / max(support.std(), 1e-8))
    if len({len(h) for h in histories}) != 1:
        raise ValueError('Support histories have different lengths; choose an alignment rule before clustering')
    labels = KMeans(n_clusters=3, n_init=50, random_state=SEED).fit_predict(np.stack(histories))
    return pd.DataFrame({'site': sorted(sites), 'Level_0': labels})


def show_folds(clusters):
    mapping = dict(zip(clusters['site'], clusters['Level_0']))
    for fold in range(5):
        train, val, test = split_sites_fivefold(clusters['site'], mapping, fold, SEED)
        print(f'Fold {fold}: train={len(train)}, val={len(val)}, test={len(test)}')
        print('  train:', train)
        print('  val:  ', val)
        print('  test: ', test)


In [ ]:
# Sediment: NGR → MS; ancient support, younger query.
ngr_ms_clusters = cluster_sites('dataset/sediment/NGR.csv', 'dataset/sediment/MS.csv', time_step=10000)
show_folds(ngr_ms_clusters)
out = PROJECT_ROOT / 'cluster/ngr/ngr_cluster.csv'
out.parent.mkdir(parents=True, exist_ok=True)
ngr_ms_clusters.to_csv(out, index=False)


In [ ]:
# Sediment: PWL → MS; ancient support, younger query.
pwl_ms_clusters = cluster_sites('dataset/sediment/PWL.csv', 'dataset/sediment/MS.csv', time_step=10000)
show_folds(pwl_ms_clusters)
out = PROJECT_ROOT / 'cluster/pwl/pwl_cluster.csv'
out.parent.mkdir(parents=True, exist_ok=True)
pwl_ms_clusters.to_csv(out, index=False)
